# Almost Ramannet version

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Import project modules
from config import STRAINS, ATCC_GROUPINGS, antibiotics, HIERARCHICAL_GROUPINGS, HIERARCHICAL_CLASSES

def set_seed(seed=123):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(123)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), '..', 'data')
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Load the dataset
X_ref = np.load(os.path.join(DATA_DIR, "X_reference.npy"))
y_ref = np.load(os.path.join(DATA_DIR, "y_reference.npy")).astype(int)
X_te = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_te = np.load(os.path.join(DATA_DIR, "y_test.npy")).astype(int)

print("Dataset shapes:")
print(f"X_reference: {X_ref.shape}")
print(f"y_reference: {y_ref.shape}")
print(f"X_test: {X_te.shape}")
print(f"y_test: {y_te.shape}")

# Check strain distribution
print(f"\nStrain distribution in reference set:")
strain_counts = Counter(y_ref)
for strain_id, count in sorted(strain_counts.items()):
    print(f"Strain {strain_id} ({STRAINS[strain_id]}): {count} samples")

In [ ]:
def get_hierarchical_strains():
    """Get strain IDs that belong to MSSA or MRSA classes"""
    hierarchical_strains = [14, 15, 16, 17, 18]  # MSSA and MRSA strains
    return sorted(hierarchical_strains)

# Get focused strains for hierarchical classification
focused_strains = get_hierarchical_strains()
print("\nHIERARCHICAL labels: strain -> superclass mapping")

for sid in focused_strains:
    superclass_id = HIERARCHICAL_GROUPINGS[sid]
    strain_name = STRAINS[sid]
    superclass_name = HIERARCHICAL_CLASSES[superclass_id]
    print(f"{sid:2d} | {strain_name:<20} : [{superclass_id}] {superclass_name}")

In [ ]:

# Get the test data for the 5 strains
focused_strains = [14, 15, 16, 17, 18]  # MSSA and MRSA strains

# Filter test data for these strains
te_mask = np.isin(y_te, focused_strains)
X_te_focused = X_te[te_mask]
y_te_focused = y_te[te_mask]

print(f"Test data shapes:")
print(f"X_te_focused: {X_te_focused.shape}")
print(f"y_te_focused: {y_te_focused.shape}")

# Plot mean spectra for each strain
plt.figure(figsize=(15, 10))

colors = ['blue', 'green', 'red', 'orange', 'purple']
strain_names = ['MSSA 1', 'MSSA 3', 'MRSA 1', 'MRSA 2', 'MSSA 2']

for i, strain_id in enumerate(focused_strains):
    # Get data for this strain
    strain_mask = y_te_focused == strain_id
    strain_data = X_te_focused[strain_mask]
    
    if len(strain_data) > 0:
        # Calculate mean spectrum
        mean_spectrum = np.mean(strain_data, axis=0)
        std_spectrum = np.std(strain_data, axis=0)
        
        # Plot mean spectrum
        plt.subplot(2, 3, i+1)
        plt.plot(mean_spectrum, color=colors[i], linewidth=1, alpha=0.8)
        plt.fill_between(range(len(mean_spectrum)), 
                        mean_spectrum - std_spectrum, 
                        mean_spectrum + std_spectrum, 
                        alpha=0.3, color=colors[i])
        plt.title(f'Strain {strain_id} ({strain_names[i]})\nn={len(strain_data)} samples')
        plt.xlabel('Wavenumber Index')
        plt.ylabel('Intensity')
        plt.grid(True, alpha=0.3)

# Plot all mean spectra together for comparison
plt.subplot(2, 3, 6)
for i, strain_id in enumerate(focused_strains):
    strain_mask = y_te_focused == strain_id
    strain_data = X_te_focused[strain_mask]
    
    if len(strain_data) > 0:
        mean_spectrum = np.mean(strain_data, axis=0)
        plt.plot(mean_spectrum, color=colors[i], linewidth=2, 
                label=f'Strain {strain_id} ({strain_names[i]})', alpha=0.8)

plt.title('All Mean Spectra Comparison')
plt.xlabel('Wavenumber Index')
plt.ylabel('Mean Intensity')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate correlations between mean spectra
print("\n=== SPECTRAL CORRELATION ANALYSIS ===")
mean_spectra = {}
for i, strain_id in enumerate(focused_strains):
    strain_mask = y_te_focused == strain_id
    strain_data = X_te_focused[strain_mask]
    
    if len(strain_data) > 0:
        mean_spectra[strain_id] = np.mean(strain_data, axis=0)

# Calculate pairwise correlations
print("\nPairwise correlations between mean spectra:")
for i, strain1 in enumerate(focused_strains):
    for j, strain2 in enumerate(focused_strains):
        if i < j and strain1 in mean_spectra and strain2 in mean_spectra:
            corr = np.corrcoef(mean_spectra[strain1], mean_spectra[strain2])[0, 1]
            print(f"Strain {strain1} vs Strain {strain2}: {corr:.4f}")

# Calculate MSSA vs MRSA group correlations
print("\n=== MSSA vs MRSA GROUP ANALYSIS ===")
mssa_strains = [14, 15, 18]  # MSSA 1, 3, 2
mrsa_strains = [16, 17]       # MRSA 1, 2

# Calculate mean spectrum for MSSA group
mssa_data = []
for strain_id in mssa_strains:
    strain_mask = y_te_focused == strain_id
    if np.any(strain_mask):
        mssa_data.append(X_te_focused[strain_mask])

if mssa_data:
    mssa_combined = np.vstack(mssa_data)
    mssa_mean = np.mean(mssa_combined, axis=0)
    print(f"MSSA group: {len(mssa_combined)} samples")

# Calculate mean spectrum for MRSA group
mrsa_data = []
for strain_id in mrsa_strains:
    strain_mask = y_te_focused == strain_id
    if np.any(strain_mask):
        mrsa_data.append(X_te_focused[strain_mask])

if mrsa_data:
    mrsa_combined = np.vstack(mrsa_data)
    mrsa_mean = np.mean(mrsa_combined, axis=0)
    print(f"MRSA group: {len(mrsa_combined)} samples")

# Calculate correlation between MSSA and MRSA groups
if mssa_data and mrsa_data:
    group_corr = np.corrcoef(mssa_mean, mrsa_mean)[0, 1]
    print(f"\nMSSA vs MRSA group correlation: {group_corr:.4f}")
    
    if group_corr > 0.99:
        print("  WARNING: MSSA and MRSA groups are nearly identical!")
        print("This explains the low classification accuracy.")
    elif group_corr > 0.95:
        print("  CAUTION: MSSA and MRSA groups are very similar.")
        print("This may explain the moderate classification accuracy.")
    else:
        print(" MSSA and MRSA groups show some differences.")
        print("Classification should be possible with better methods.")

# Plot MSSA vs MRSA comparison
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
if mssa_data and mrsa_data:
    plt.plot(mssa_mean, color='blue', linewidth=2, label='MSSA Group Mean')
    plt.plot(mrsa_mean, color='red', linewidth=2, label='MRSA Group Mean')
    plt.title(f'MSSA vs MRSA Group Comparison\nCorrelation: {group_corr:.4f}')
    plt.xlabel('Wavenumber Index')
    plt.ylabel('Mean Intensity')
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
if mssa_data and mrsa_data:
    # Plot difference spectrum
    diff_spectrum = mssa_mean - mrsa_mean
    plt.plot(diff_spectrum, color='green', linewidth=2)
    plt.title('Difference Spectrum (MSSA - MRSA)')
    plt.xlabel('Wavenumber Index')
    plt.ylabel('Intensity Difference')
    plt.grid(True, alpha=0.3)
    
    # Highlight significant differences
    threshold = np.std(diff_spectrum) * 2
    significant_diffs = np.abs(diff_spectrum) > threshold
    plt.fill_between(range(len(diff_spectrum)), 
                    -threshold, threshold, 
                    alpha=0.2, color='gray', label=f'±2σ threshold')
    plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f"\n=== SUMMARY ===")
print(f"Total test samples for 5 strains: {len(X_te_focused)}")
print(f"Strain distribution:")
for strain_id in focused_strains:
    count = np.sum(y_te_focused == strain_id)
    strain_name = STRAINS[strain_id]
    print(f"  Strain {strain_id} ({strain_name}): {count} samples")

In [ ]:
STRAIN_ORDER = np.arange(30)  
sid_ref_all = np.repeat(STRAIN_ORDER, 2000)
sid_te_all  = np.repeat(STRAIN_ORDER, 100)

focused_strains = get_hierarchical_strains()
# Create hierarchical class mapping
HIERARCHICAL_TO_BIN = {0: 0, 1: 1}  # MSSA=0, MRSA=1
BIN_TO_HIERARCHICAL = {0: 0, 1: 1}

# Filter data for hierarchical classification
ref_mask = np.isin(sid_ref_all, focused_strains)
te_mask = np.isin(sid_te_all, focused_strains)

sid_ref_filtered = sid_ref_all[ref_mask]
sid_te_filtered = sid_te_all[te_mask]

X_ref_filtered = X_ref[ref_mask]
X_te_filtered = X_te[te_mask]

# Create hierarchical labels
y_ref_filtered = np.array([HIERARCHICAL_TO_BIN[HIERARCHICAL_GROUPINGS[s]] for s in sid_ref_filtered], dtype=int)
y_te_filtered = np.array([HIERARCHICAL_TO_BIN[HIERARCHICAL_GROUPINGS[s]] for s in sid_te_filtered], dtype=int)

print(f"\nHierarchical classification data shapes:")
print(f"X_ref_filtered: {X_ref_filtered.shape}")
print(f"y_ref_filtered: {y_ref_filtered.shape}")
print(f"X_te_filtered: {X_te_filtered.shape}")
print(f"y_te_filtered: {y_te_filtered.shape}")

# Show class distribution
print(f"\nClass distribution in hierarchical dataset:")
from collections import Counter
class_counts = Counter(y_ref_filtered)
for class_id, count in sorted(class_counts.items()):
    class_name = HIERARCHICAL_CLASSES[class_id]
    print(f"Class {class_id} ({class_name}): {count} samples")

In [ ]:
# Get the focused strains (MSSA and MRSA only)
focused_strains = get_hierarchical_strains()

# Separate into MSSA and MRSA groups
mssa_strains = [s for s in focused_strains if HIERARCHICAL_GROUPINGS[s] == 0]
mrsa_strains = [s for s in focused_strains if HIERARCHICAL_GROUPINGS[s] == 1]

# Create reordering indices - MSSA first, then MRSA
reorder_indices = []
for strain_id in mssa_strains:
    # Find all indices where y_te_focused equals this strain_id
    strain_indices = np.where(y_te_focused == strain_id)[0]
    reorder_indices.extend(strain_indices)

for strain_id in mrsa_strains:
    # Find all indices where y_te_focused equals this strain_id
    strain_indices = np.where(y_te_focused == strain_id)[0]
    reorder_indices.extend(strain_indices)

# Reorder the data according to focused strains
X_te_reordered = X_te_focused[reorder_indices]
y_te_reordered = y_te_focused[reorder_indices]

# Get hierarchical class labels for the reordered data
y_te_class_reordered = np.array([HIERARCHICAL_GROUPINGS[strain_id] for strain_id in y_te_reordered])

# Plot the focused spectra as a heatmap
plt.figure(figsize=(15, 8))
plt.imshow(X_te_reordered, cmap='viridis', aspect='auto', interpolation='nearest')
plt.title('MSSA and MRSA Test Spectra - Grouped by Class')
plt.xlabel('Wavenumber Index (1000 points)')
plt.ylabel('Spectrum Index')

# Add y-axis labels showing strain and class info
yticks = np.arange(0, len(y_te_class_reordered), 100)
ytick_labels = []
for i in yticks:
    if i < len(y_te_reordered):
        strain_id = y_te_reordered[i]
        class_id = y_te_class_reordered[i]
        strain_name = STRAINS[strain_id]
        class_name = HIERARCHICAL_CLASSES[class_id]
        ytick_labels.append(f'{i}: {strain_id} ({strain_name}) - {class_name}')

plt.yticks(yticks, ytick_labels, rotation=0, fontsize=8)

plt.colorbar(label='Intensity')
plt.tight_layout()
plt.show()

# Print the reordering information
print("Focused strains (MSSA and MRSA only):")
print("MSSA strains =", mssa_strains)
print("MRSA strains =", mrsa_strains)
print(f"Total spectra in focused set: {len(reorder_indices)}")
print("First 10 strain IDs in new order:", y_te_reordered[:10])
print("First 10 class labels in new order:", y_te_class_reordered[:10])

# Show strain distribution
print("\nStrain distribution in focused test set:")
from collections import Counter
strain_counts = Counter(y_te_reordered)
for strain_id, count in sorted(strain_counts.items()):
    class_id = HIERARCHICAL_GROUPINGS[strain_id]
    print(f"Strain {strain_id} ({STRAINS[strain_id]}) - {HIERARCHICAL_CLASSES[class_id]}: {count} samples")

In [ ]:
# Get the focused strains (MSSA and MRSA only)
focused_strains = get_hierarchical_strains()

# Separate into MSSA and MRSA groups
mssa_strains = [s for s in focused_strains if HIERARCHICAL_GROUPINGS[s] == 0]
mrsa_strains = [s for s in focused_strains if HIERARCHICAL_GROUPINGS[s] == 1]

# Sort MSSA strains in order: MSSA 1 (14), MSSA 2 (18), MSSA 3 (15)
mssa_sorted = [14, 18, 15]  # MSSA 1, 2, 3
mssa_sorted = [s for s in mssa_sorted if s in mssa_strains]

# Sort MRSA strains in order: MRSA 1 (16), MRSA 2 (17)
mrsa_sorted = [16, 17]  # MRSA 1, 2
mrsa_sorted = [s for s in mrsa_sorted if s in mrsa_strains]

# Create reordering indices - MSSA first (1,2,3), then MRSA (1,2)
reorder_indices = []
for strain_id in mssa_sorted:
    # Find all indices where y_te_focused equals this strain_id
    strain_indices = np.where(y_te_focused == strain_id)[0]
    reorder_indices.extend(strain_indices)

for strain_id in mrsa_sorted:
    # Find all indices where y_te_focused equals this strain_id
    strain_indices = np.where(y_te_focused == strain_id)[0]
    reorder_indices.extend(strain_indices)

# Reorder the data according to focused strains
X_te_reordered = X_te_focused[reorder_indices]
y_te_reordered = y_te_focused[reorder_indices]

# Get hierarchical class labels for the reordered data
y_te_class_reordered = np.array([HIERARCHICAL_GROUPINGS[strain_id] for strain_id in y_te_reordered])

# Helper function to clean strain name (remove "isogenic" and avoid repeating class name)
def get_clean_strain_label(strain_id, class_name):
    """Get clean strain label without repeating class name and without 'isogenic'"""
    strain_name = STRAINS[strain_id]
    # Remove "(isogenic)" if present
    strain_name = strain_name.replace(" (isogenic)", "")
    # If strain name already contains the class name, don't repeat it
    if class_name in strain_name:
        return strain_name
    else:
        return f"{strain_name} - {class_name}"

# Plot the focused spectra as a heatmap
plt.figure(figsize=(15, 8))
plt.imshow(X_te_reordered, cmap='viridis', aspect='auto', interpolation='nearest')
# Removed title
plt.xlabel('Wavenumber Index (1000 points)', fontsize=14)
plt.ylabel('Spectrum Index', fontsize=14, labelpad=20)

# Add y-axis labels showing strain and class info (without repeating class name)
yticks = np.arange(0, len(y_te_class_reordered), 100)
ytick_labels = []
for i in yticks:
    if i < len(y_te_reordered):
        strain_id = y_te_reordered[i]
        class_id = y_te_class_reordered[i]
        class_name = HIERARCHICAL_CLASSES[class_id]
        clean_label = get_clean_strain_label(strain_id, class_name)
        ytick_labels.append(f'{i}: {clean_label}')

plt.yticks(yticks, ytick_labels, rotation=0, fontsize=12)
plt.xticks(fontsize=12)

cbar = plt.colorbar(label='Intensity')
cbar.set_label('Intensity', fontsize=14)
cbar.ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()

# Print the reordering information
print("Focused strains (MSSA and MRSA only):")
print("MSSA strains =", mssa_sorted)
print("MRSA strains =", mrsa_sorted)
print(f"Total spectra in focused set: {len(reorder_indices)}")
print("First 10 strain IDs in new order:", y_te_reordered[:10])
print("First 10 class labels in new order:", y_te_class_reordered[:10])

# Show strain distribution
print("\nStrain distribution in focused test set:")
from collections import Counter
strain_counts = Counter(y_te_reordered)
for strain_id in mssa_sorted + mrsa_sorted:
    if strain_id in strain_counts:
        count = strain_counts[strain_id]
        class_id = HIERARCHICAL_GROUPINGS[strain_id]
        class_name = HIERARCHICAL_CLASSES[class_id]
        clean_label = get_clean_strain_label(strain_id, class_name)
        print(f"{clean_label}: {count} samples")

In [ ]:
# Get the focused strains (MSSA and MRSA only)
focused_strains = get_hierarchical_strains()

# Separate into MSSA and MRSA groups
mssa_strains = [s for s in focused_strains if HIERARCHICAL_GROUPINGS[s] == 0]
mrsa_strains = [s for s in focused_strains if HIERARCHICAL_GROUPINGS[s] == 1]

# Sort MSSA strains in order: MSSA 1 (14), MSSA 2 (18), MSSA 3 (15)
mssa_sorted = [14, 18, 15]  # MSSA 1, 2, 3
mssa_sorted = [s for s in mssa_sorted if s in mssa_strains]

# Sort MRSA strains in order: MRSA 1 (16), MRSA 2 (17)
mrsa_sorted = [16, 17]  # MRSA 1, 2
mrsa_sorted = [s for s in mrsa_sorted if s in mrsa_strains]

# Create reordering indices - MSSA first (1,2,3), then MRSA (1,2)
reorder_indices = []
for strain_id in mssa_sorted:
    # Find all indices where y_te_focused equals this strain_id
    strain_indices = np.where(y_te_focused == strain_id)[0]
    reorder_indices.extend(strain_indices)

for strain_id in mrsa_sorted:
    # Find all indices where y_te_focused equals this strain_id
    strain_indices = np.where(y_te_focused == strain_id)[0]
    reorder_indices.extend(strain_indices)

# Reorder the data according to focused strains
X_te_reordered = X_te_focused[reorder_indices]
y_te_reordered = y_te_focused[reorder_indices]

# Get hierarchical class labels for the reordered data
y_te_class_reordered = np.array([HIERARCHICAL_GROUPINGS[strain_id] for strain_id in y_te_reordered])

# Helper function to clean strain name (remove "isogenic" and avoid repeating class name)
def get_clean_strain_label(strain_id, class_name):
    """Get clean strain label without repeating class name and without 'isogenic'"""
    strain_name = STRAINS[strain_id]
    # Remove "(isogenic)" if present
    strain_name = strain_name.replace(" (isogenic)", "")
    # If strain name already contains the class name, don't repeat it
    if class_name in strain_name:
        return strain_name
    else:
        return f"{strain_name} - {class_name}"

# Plot the focused spectra as a heatmap
plt.figure(figsize=(15, 8))
plt.imshow(X_te_reordered, cmap='viridis', aspect='auto', interpolation='nearest')
# Removed title
plt.xlabel('Wavenumber Index (1000 points)', fontsize=18, fontweight='bold')
plt.ylabel('Spectrum Index', fontsize=18, fontweight='bold', labelpad=20)

# Add y-axis labels showing strain and class info (without repeating class name)
yticks = np.arange(0, len(y_te_class_reordered), 100)
ytick_labels = []
for i in yticks:
    if i < len(y_te_reordered):
        strain_id = y_te_reordered[i]
        class_id = y_te_class_reordered[i]
        class_name = HIERARCHICAL_CLASSES[class_id]
        clean_label = get_clean_strain_label(strain_id, class_name)
        ytick_labels.append(f'{i}: {clean_label}')

plt.yticks(yticks, ytick_labels, rotation=0, fontsize=16)
plt.xticks(fontsize=16)
# Make tick labels bold
for label in plt.gca().get_yticklabels():
    label.set_fontweight('bold')
for label in plt.gca().get_xticklabels():
    label.set_fontweight('bold')

cbar = plt.colorbar(label='Intensity')
cbar.set_label('Intensity', fontsize=18, fontweight='bold')
cbar.ax.tick_params(labelsize=16)
# Make colorbar tick labels bold
for label in cbar.ax.get_yticklabels():
    label.set_fontweight('bold')

plt.tight_layout()
plt.show()

# Print the reordering information
print("Focused strains (MSSA and MRSA only):")
print("MSSA strains =", mssa_sorted)
print("MRSA strains =", mrsa_sorted)
print(f"Total spectra in focused set: {len(reorder_indices)}")
print("First 10 strain IDs in new order:", y_te_reordered[:10])
print("First 10 class labels in new order:", y_te_class_reordered[:10])

# Show strain distribution
print("\nStrain distribution in focused test set:")
from collections import Counter
strain_counts = Counter(y_te_reordered)
for strain_id in mssa_sorted + mrsa_sorted:
    if strain_id in strain_counts:
        count = strain_counts[strain_id]
        class_id = HIERARCHICAL_GROUPINGS[strain_id]
        class_name = HIERARCHICAL_CLASSES[class_id]
        clean_label = get_clean_strain_label(strain_id, class_name)
        print(f"{clean_label}: {count} samples")

In [ ]:
# Get the focused strains (MSSA and MRSA only)
focused_strains = get_hierarchical_strains()

# Separate into MSSA and MRSA groups
mssa_strains = [s for s in focused_strains if HIERARCHICAL_GROUPINGS[s] == 0]
mrsa_strains = [s for s in focused_strains if HIERARCHICAL_GROUPINGS[s] == 1]

# Sort MSSA strains in order: MSSA 1 (14), MSSA 2 (18), MSSA 3 (15)
mssa_sorted = [14, 18, 15]  # MSSA 1, 2, 3
mssa_sorted = [s for s in mssa_sorted if s in mssa_strains]

# Sort MRSA strains in order: MRSA 1 (16), MRSA 2 (17)
mrsa_sorted = [16, 17]  # MRSA 1, 2
mrsa_sorted = [s for s in mrsa_sorted if s in mrsa_strains]

# Create reordering indices - MSSA first (1,2,3), then MRSA (1,2)
reorder_indices = []
for strain_id in mssa_sorted:
    # Find all indices where y_te_focused equals this strain_id
    strain_indices = np.where(y_te_focused == strain_id)[0]
    reorder_indices.extend(strain_indices)

for strain_id in mrsa_sorted:
    # Find all indices where y_te_focused equals this strain_id
    strain_indices = np.where(y_te_focused == strain_id)[0]
    reorder_indices.extend(strain_indices)

# Reorder the data according to focused strains
X_te_reordered = X_te_focused[reorder_indices]
y_te_reordered = y_te_focused[reorder_indices]

# Get hierarchical class labels for the reordered data
y_te_class_reordered = np.array([HIERARCHICAL_GROUPINGS[strain_id] for strain_id in y_te_reordered])

# Helper function to clean strain name (remove "isogenic" and avoid repeating class name)
def get_clean_strain_label(strain_id, class_name):
    """Get clean strain label without repeating class name and without 'isogenic'"""
    strain_name = STRAINS[strain_id]
    # Remove "(isogenic)" if present
    strain_name = strain_name.replace(" (isogenic)", "")
    # If strain name already contains the class name, don't repeat it
    if class_name in strain_name:
        return strain_name
    else:
        return f"{strain_name} - {class_name}"

# Plot the focused spectra as a heatmap
plt.figure(figsize=(15, 8))
plt.imshow(X_te_reordered, cmap='viridis', aspect='auto', interpolation='nearest')
# Removed title
plt.xlabel('Wavenumber Index (1000 points)', fontsize=20, fontweight='bold')
plt.ylabel('Spectrum Index', fontsize=20, fontweight='bold', labelpad=20)

# Add y-axis labels showing strain and class info (without repeating class name)
yticks = np.arange(0, len(y_te_class_reordered), 100)
ytick_labels = []
for i in yticks:
    if i < len(y_te_reordered):
        strain_id = y_te_reordered[i]
        class_id = y_te_class_reordered[i]
        class_name = HIERARCHICAL_CLASSES[class_id]
        clean_label = get_clean_strain_label(strain_id, class_name)
        ytick_labels.append(f'{i}: {clean_label}')

plt.yticks(yticks, ytick_labels, rotation=0, fontsize=20)
plt.xticks(fontsize=20)
# Make tick labels bold
for label in plt.gca().get_yticklabels():
    label.set_fontweight('bold')
for label in plt.gca().get_xticklabels():
    label.set_fontweight('bold')

cbar = plt.colorbar(label='Intensity')
cbar.set_label('Intensity', fontsize=20, fontweight='bold')
cbar.ax.tick_params(labelsize=20)
# Make colorbar tick labels bold
for label in cbar.ax.get_yticklabels():
    label.set_fontweight('bold')

plt.tight_layout()
plt.show()

# Print the reordering information
print("Focused strains (MSSA and MRSA only):")
print("MSSA strains =", mssa_sorted)
print("MRSA strains =", mrsa_sorted)
print(f"Total spectra in focused set: {len(reorder_indices)}")
print("First 10 strain IDs in new order:", y_te_reordered[:10])
print("First 10 class labels in new order:", y_te_class_reordered[:10])

# Show strain distribution
print("\nStrain distribution in focused test set:")
from collections import Counter
strain_counts = Counter(y_te_reordered)
for strain_id in mssa_sorted + mrsa_sorted:
    if strain_id in strain_counts:
        count = strain_counts[strain_id]
        class_id = HIERARCHICAL_GROUPINGS[strain_id]
        class_name = HIERARCHICAL_CLASSES[class_id]
        clean_label = get_clean_strain_label(strain_id, class_name)
        print(f"{clean_label}: {count} samples")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class RamanNetResidualBlock(nn.Module):
    """
    Residual block for RamanNet architecture as described in the paper.
    Each residual layer contains a shortcut connection between input and output
    of one convolutional layer.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(RamanNetResidualBlock, self).__init__()
        
        # Single convolutional layer as per paper description
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, 
                             padding=kernel_size//2, bias=False)
        self.bn = nn.BatchNorm1d(out_channels)
        
        # Shortcut connection
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm1d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn(self.conv(x)))
        # FIXED: Use out = out + self.shortcut(x) instead of += to avoid in-place operation
        out = out + self.shortcut(x)
        out = F.relu(out)
        return out


class BinaryClassifier(nn.Module):
    """
    BinaryClassifier model following the exact RamanNet architecture described in the paper.
    
    Architecture:
    1. Initial convolution layer (kernel size 7)
    2. Two residual layers (kernel sizes 7 and 3)
    3. Flatten layer
    4. Final fully connected classification layer (2 classes for binary classification)
    
    All convolution kernel numbers are set to 1 as specified in the paper.
    Uses kaiming normal initialization.
    """
    def __init__(self, input_dim=1000, n_classes=2):
        super(BinaryClassifier, self).__init__()
        self.input_dim = input_dim
        self.n_classes = n_classes
        
        # Initial convolution layer (kernel size 7, 1 channel)
        self.conv1 = nn.Conv1d(1, 1, kernel_size=7, stride=1, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(1)
        
        # First residual layer (kernel size 7)
        self.residual1 = RamanNetResidualBlock(1, 1, kernel_size=7)
        
        # Second residual layer (kernel size 3)
        self.residual2 = RamanNetResidualBlock(1, 1, kernel_size=3)
        
        # Flatten layer
        self.flatten = nn.Flatten()
        
        # Final fully connected classification layer
        self.classifier = nn.Linear(input_dim, n_classes)
        
        # Initialize weights using kaiming normal initialization
        self._initialize_weights()

    def forward(self, x):
        # Initial convolution layer
        x = F.relu(self.bn1(self.conv1(x)))
        
        # Two residual layers
        x = self.residual1(x)
        x = self.residual2(x)
        
        # Flatten
        x = self.flatten(x)
        
        # Final classification layer
        x = self.classifier(x)
        
        return x

    def _initialize_weights(self):
        """
        Initialize weights using kaiming normal initialization as specified in the paper.
        """
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.constant_(m.bias, 0)

In [ ]:
def prepare_data(X, y, batch_size=32, shuffle=True):
    """Convert numpy arrays to DataLoader with shape [B,1,L]."""
    X_tensor = torch.as_tensor(X, dtype=torch.float32).unsqueeze(1)  # [N,1,L]
    y_tensor = torch.as_tensor(y, dtype=torch.long)
    dataset  = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False)

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for data, target in train_loader:
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad(set_to_none=True)
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        # sample-weighted loss to avoid bias from a short last batch
        bs = target.size(0)
        total_loss += loss.detach().item() * bs
        pred = output.detach().argmax(dim=1)
        correct += (pred == target).sum().item()
        total += bs

    avg_loss = total_loss / max(total, 1)
    acc = 100.0 * correct / max(total, 1)
    return avg_loss, acc

In [ ]:
# Evaluation function
def evaluate(model, test_loader, criterion, device):
    model.eval()
    assert not model.training, "Model must be in eval() during evaluation."

    total_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)

            bs = target.size(0)
            total_loss += loss.detach().item() * bs
            pred = output.detach().argmax(dim=1)

            correct += (pred == target).sum().item()
            total += bs
            all_preds.extend(pred.cpu().tolist())
            all_targets.extend(target.cpu().tolist())

    avg_loss = total_loss / max(total, 1)
    acc = 100.0 * correct / max(total, 1)
    return avg_loss, acc, all_preds, all_targets

In [ ]:
def train_eval_permutation_with_epochs(
    X_train, y_train, X_test, y_test,
    epochs=10, lr=1.5e-3, batch_size=256, weight_decay=1e-1
):
    
    torch.manual_seed(123)
    torch.cuda.manual_seed_all(123)
    
    # Infer spectrum length from data
    input_dim = X_train.shape[1]
    assert X_test.shape[1] == input_dim, "Train/Test spectrum length mismatch."

    # Model (RamanNet) + loaders
    model = BinaryClassifier(input_dim=input_dim, n_classes=2).to(DEVICE)
    train_loader = prepare_data(X_train, y_train, batch_size=batch_size, shuffle=True)
    test_loader  = prepare_data(X_test,  y_test,  batch_size=batch_size, shuffle=False)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_test_acc = -1.0
    best_epoch = -1
    best_state = None

    for epoch in range(1, epochs + 1):
        # ---- Train ----
        model.train()
        running_correct, running_total, running_loss = 0, 0, 0.0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)  # [B,1,L]
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.detach().item() * xb.size(0)
            preds = logits.argmax(dim=1)
            running_correct += (preds == yb).sum().item()
            running_total += xb.size(0)

        train_acc = 100.0 * running_correct / max(1, running_total)

        # ---- Eval ----
        model.eval()
        test_correct, test_total, test_loss_sum = 0, 0, 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)

                logits = model(xb)
                loss = criterion(logits, yb)

                test_loss_sum += float(loss) * xb.size(0)
                preds = logits.argmax(dim=1)
                test_correct += (preds == yb).sum().item()
                test_total += xb.size(0)

        test_acc = 100.0 * test_correct / max(1, test_total)
        print(f"    Epoch {epoch}: Train Acc = {train_acc:.2f}%, Test Acc = {test_acc:.2f}%")

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"[BEST] Epoch {best_epoch}: Test Acc = {best_test_acc:.2f}%")
    return best_test_acc / 100.0

In [ ]:
LAST_PERM_MAP = None
from itertools import combinations


def generate_all_unique_permutations(focused_strains):
    """
    Generate all 10 unique ways to assign 3 strains to MSSA and 2 to MRSA
    """
    n_mssa = sum(HIERARCHICAL_GROUPINGS[s] == 0 for s in focused_strains)  # 3
    n_mrsa = sum(HIERARCHICAL_GROUPINGS[s] == 1 for s in focused_strains)  # 2
    
    # Generate all combinations of 3 strains for MSSA
    mssa_combinations = list(combinations(focused_strains, n_mssa))
    
    unique_permutations = []
    for mssa_strains in mssa_combinations:
        mrsa_strains = [s for s in focused_strains if s not in mssa_strains]
        
        # Create strain-to-label mapping
        strain_to_label = {}
        for strain in mssa_strains:
            strain_to_label[strain] = HIERARCHICAL_TO_BIN[0]  # MSSA = 0
        for strain in mrsa_strains:
            strain_to_label[strain] = HIERARCHICAL_TO_BIN[1]  # MRSA = 1
            
        unique_permutations.append(strain_to_label)
    
    return unique_permutations

def print_hierarchical_permutation_mapping(perm_idx, y_ref_perm, y_te_perm):
    print(f"\nPermutation {perm_idx}: strain -> hierarchical class (TRUE -> PERMUTED)")
    if LAST_PERM_MAP is None:
        print("  [warn] LAST_PERM_MAP is not set")
        return
    for sid in focused_strains:
        true_superclass = HIERARCHICAL_GROUPINGS[sid]             
        perm_bin = LAST_PERM_MAP[sid]             
        perm_superclass = BIN_TO_HIERARCHICAL[perm_bin]         
        strain_name = STRAINS[sid]
        true_name = HIERARCHICAL_CLASSES[true_superclass]
        perm_name = HIERARCHICAL_CLASSES[perm_superclass]
        print(f"{sid:2d} | {strain_name:<20} : [{true_superclass}] {true_name}  ->  [{perm_superclass}] {perm_name}")

In [ ]:
# Permutation test parameters
#N_PERMUTATIONS = 1000  # Number of permutations
PERM_EPOCHS = 10   # Epochs for each permutation
RANDOM_SEED =42

In [ ]:
# Train with true hierarchical labels
print(f"\nTRUE hierarchical labels training:")
true_accuracy = train_eval_permutation_with_epochs(
    X_ref_filtered, y_ref_filtered,
    X_te_filtered, y_te_filtered,
    epochs=PERM_EPOCHS
)
print(f"TRUE hierarchical labels final accuracy: {true_accuracy:.2%}")


In [ ]:

null_accuracies = []


ref_strains_order = sid_ref_filtered   
te_strains_order = sid_te_filtered    

# Generate all unique permutations
unique_permutations = generate_all_unique_permutations(focused_strains)

for perm_idx, strain_to_label in enumerate(unique_permutations, 1):
    # Apply the same mapping to both splits
    y_ref_permuted = np.array([strain_to_label[s] for s in ref_strains_order], dtype=int)
    y_te_permuted = np.array([strain_to_label[s] for s in te_strains_order], dtype=int)
    
    # Print the mapping
    print(f"\nPermutation {perm_idx}: strain -> hierarchical class (TRUE -> PERMUTED)")
    for sid in focused_strains:
        true_superclass = HIERARCHICAL_GROUPINGS[sid]
        perm_bin = strain_to_label[sid]
        perm_superclass = BIN_TO_HIERARCHICAL[perm_bin]
        strain_name = STRAINS[sid]
        true_name = HIERARCHICAL_CLASSES[true_superclass]
        perm_name = HIERARCHICAL_CLASSES[perm_superclass]
        print(f"{sid:2d} | {strain_name:<20} : [{true_superclass}] {true_name}  ->  [{perm_superclass}] {perm_name}")
    
    # Train & evaluate with permuted labels
    print(f"\nPermutation {perm_idx} training:")
    perm_accuracy = train_eval_permutation_with_epochs(
        X_ref_filtered, y_ref_permuted,
        X_te_filtered, y_te_permuted,
        epochs=PERM_EPOCHS
    )
    null_accuracies.append(perm_accuracy)
    print(f"Permutation {perm_idx} final accuracy: {perm_accuracy:.2%}")

print("Unique hierarchical permutation test completed!")

In [ ]:
# Create histogram visualization for hierarchical permutation test
null_accuracies = np.array(null_accuracies)

plt.figure(figsize=(7, 5))
plt.hist(null_accuracies * 100, bins=20, color='navy', alpha=0.7)

# Observed performance line
plt.axvline(true_accuracy * 100, color='red', linewidth=2, label='True Performance')

# Chance baseline for binary classification = 50%
plt.axvline(50, color='k', linestyle=':', linewidth=1, label='Chance (50%)')

plt.xlabel("Accuracy [%] (on the test set)")
plt.ylabel("Frequency")
plt.title("Permutation null vs observed (Hierarchical: MSSA vs MRSA)")

# Add text annotation with observed accuracy and permutation p-value
ymax = max(np.histogram(null_accuracies * 100, bins=20)[0])
p_value = np.mean(null_accuracies >= true_accuracy)

plt.text(true_accuracy * 100 + 2, ymax * 0.6,
         f"Observed = {true_accuracy * 100:.2f}%\nP = {p_value:.4f}",
         fontsize=12, bbox=dict(facecolor='yellow', alpha=0.7))

plt.legend()
plt.grid(True, alpha=0.3)
plt.show()